In [ ]:
import importlib

if importlib.util.find_spec("easyocr") is None:
    !pip install docling langchain-docling langchain-community pymupdf flask easyocr
else:
    print("Packages already installed, skipping...")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from langchain_docling.loader import DoclingLoader
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorDevice, EasyOcrOptions, AcceleratorOptions
from docling.document_converter import PdfFormatOption

from langchain_community.document_loaders import PyMuPDFLoader

from pathlib import Path
from flask import Flask, jsonify

import threading
import time
import json
import gc

# Declare paths
base_path = "/content/drive/MyDrive/Legal_RAG_assistant/legal_docs"
processed_path = "/content/drive/MyDrive/Legal_RAG_assistant/processed"

# Init
pipeline_options = PdfPipelineOptions(
    allow_external_plugins=True,
    do_ocr=True,
    accelerator_options=AcceleratorOptions(device=AcceleratorDevice.CUDA)
)

pipeline_options.ocr_options = EasyOcrOptions(
    lang=["vi", "en"],
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

def is_text_extractable(docs, threshold=200):
    if not docs:
        return False
    avg_chars = sum(len(d.page_content) for d in docs) / len(docs)
    return avg_chars > threshold

def load_and_process_docs(file_path):
    print(f"--> [TIMING]: Quick text check on {file_path}...")
    start_pymupdf = time.perf_counter()

    try:
        loader = PyMuPDFLoader(file_path)
        sample_docs = list(loader.load())[:10]
    except Exception as e:
        print(f"PyMuPDF failed: {e}")
        sample_docs = []

    end_pymupdf = time.perf_counter()

    if is_text_extractable(sample_docs):
        print(f"-> [load_and_process_docs]: Native PDF detected (checked in {end_pymupdf - start_pymupdf:.4f}s). Loading full doc...")
        full_loader = PyMuPDFLoader(file_path)
        return full_loader.load()

    # Fallback: Using OCR for scanned PDF file
    docs = []
    try:
        print(f"-> [load_and_process_docs]: Scanned PDF detected. Starting Docling OCR on {file_path}. This may take a while...")
        start_ocr = time.perf_counter()

        loader = DoclingLoader(file_path, converter=converter)
        docs = list(loader.lazy_load())

        end_ocr = time.perf_counter()
        print(f"-> [load_and_process_docs]: Docling OCR finished in {end_ocr - start_ocr:.4f} seconds.")
    except Exception as e:
        print(f"-> [load_and_process_docs]: Error loading document(s) from {file_path}: {e}")

    return docs

# Process each folder
for country_folder in Path(base_path).iterdir():
    if not country_folder.is_dir():
        continue
    processed_country_folder = Path(processed_path) / country_folder.name
    processed_country_folder.mkdir(parents=True, exist_ok=True)

    # Process each pdf files
    for pdf_file in country_folder.glob("*.pdf"):
        output_file = processed_country_folder / f"{pdf_file.stem}.json"

        if output_file.exists():
            print(f"-> [load_and_process_docs]: Skipping {pdf_file.name} already processed")
            continue
        pages = load_and_process_docs(str(pdf_file))
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump({
                  "jurisdiction": country_folder.name,
                  "domain": pdf_file.stem.replace("_", " "),
                  "source": pdf_file.name,
                  "pages": [p.page_content for p in pages]
              }, f, ensure_ascii=False, indent=2)
        print(f"-> [load_and_process_docs]: {pdf_file.name} is saved")


app = Flask(__name__)

@app.route("/ingest", methods=["GET"])
def ingest():
    try:
        result = []
        for json_file in Path(processed_path).rglob("*.json"):
            with open(json_file, "r", encoding="utf-8") as f:
                result.append(json.load(f))

        print(f"-> [ingest]: Serving {len(result)} documents")
        return jsonify({"status": "ok", "documents": result})
    except Exception as e:
        print(f"-> [ingest]: Ingest failed: {e}")
        return jsonify({"status": "error", "message": str(e)})

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

print("Flask server starting up...")
threading.Thread(target=lambda: app.run(port=5000, debug=False, use_reloader=False)).start()
print("Flask server running on port 5000")

In [ ]:
import subprocess, re

# Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Start the tunnel and capture the public URL
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait until the public URL appears in the logs
for line in proc.stdout:
    print(line.strip())
    match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print(f"\nYOUR PUBLIC URL: {public_url}")
        print(f"Copy this into your local app!")
        break